In [1]:
from openai import OpenAI

In [2]:
openai_client = OpenAI()

In [3]:
import dotenv

In [4]:
dotenv.load_dotenv()

True

In [5]:
import os

Llamamos al api de responses para hacer una peticion y lo sacamos en el output_text de la respuesta

In [8]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input="tell me a joke about dogs"
)

In [9]:
print(response.output_text)

Why did the dog sit in the shade?

Because he didn't want to become a hot dog! 🐶🌭


In [ ]:
messages = [
    {"role":"user","content":"tell me a joke about cats"}
]
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input = messages
)
print(response.output_text)


Why was the cat sitting on the computer?

Because it wanted to keep an eye on the mouse!


In [11]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_0dd20414c3457bec006ab4b4277b9087d2a22ff0483857cf06",
  "created_at": 1790227495.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4o-mini-2024-07-18",
  "object": "response",
  "output": [
    {
      "id": "msg_0dd20414c3457bec006ab4b428c94c87d2aafb0fa7c94583db",
      "content": [
        {
          "annotations": [],
          "text": "Why was the cat sitting on the computer?\n\nBecause it wanted to keep an eye on the mouse!",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": null
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 1.0,
  "background": false,
  "completed_at": 1790227496.0,
  "conversation": null,
  "max_output_tokens": null,
  "max_tool_calls": null,
  "moderation": null,
  "previous_response_id": null,
  "pr

Podemos acceder a la respuesta llendo a output y eligiendo la salida y el contenido que queremos ver

In [12]:
print(response.output[0].content[0].text)

Why was the cat sitting on the computer?

Because it wanted to keep an eye on the mouse!


Ahora vamos a ver el uso

In [34]:
from openai.types.responses.response_usage import ResponseUsage
from openai.types.responses.response_usage import InputTokensDetails
from openai.types.responses.response_usage import OutputTokensDetails

In [36]:
MODEL_PRICES = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-5-nano": {"input": 0.075, "output": 0.30},
    "gpt-5-mini": {"input": 0.25, "output": 2.00},
    "gpt-5.2": {"input": 1.75, "output": 14.00},
    "gpt-5.2-pro": {"input": 21.00, "output": 168.00},
}

Es el precio por millon de tokens, cogemos el precio del modelo y dividimos los tokens de entrada y salida por 1_000_000

In [37]:
def calculate_cost(model_name: str, input_tokens: int, output_tokens: int) -> float:
    prices = MODEL_PRICES[model_name.lower()]
    input_cost = (input_tokens / 1_000_000) * prices["input"]
    output_cost = (output_tokens / 1_000_000) * prices["output"]
    return input_cost + output_cost

In [38]:
usage = response.usage

cost = calculate_cost(
    model_name="gpt-4o-mini",
    input_tokens=usage.input_tokens,
    output_tokens=usage.output_tokens
)

print(f"Cost: ${cost:.6f}")

Cost: $0.000015


Respuesstas en Streaming

Pedimos el objeto y recorremos el stream, lo que viene como delta lo sacamos y despues detectamos el objeto response

In [40]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

stream = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    stream=True
)
response = None
for event in stream:
    if hasattr(event, 'delta'):
        print(event.delta, end='')
    if hasattr(event, 'response'):
        response = event.response


Once upon a time, in a land where rainbows twisted through the sky, lived a unicorn named Sparkle. Unlike most unicorns, Sparkle was known for her clumsiness. Her dazzling horn often got her into trouble, causing mischief instead of magic.

One sunny day, Sparkle decided to explore the Enchanted Forest, a place filled with wonders and mysteries. Eager to impress her friends, she aimed to create a beautiful flower crown for herself. As she trotted through the meadows, she lost her footing on a patch of slippery moss. She stumbled and fell, accidentally knocking over a beehive.

Out came the angry bees, buzzing and swirling around in a frenzy. Sparkle panicked and tried to escape, but in her haste, she ran straight into a thicket of thorns. The thorns snagged her mane and tail, leaving her looking more like a tangled mess than the graceful creature she aspired to be.

As she thrashed to free herself, the bees followed her, murmuring in discontent. Instead of admiring her beautiful mane, 

In [41]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_0e8c5c6bc6024a90006ab4b986af3487d2b8adb442740e7d17",
  "created_at": 1790228870.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4o-mini-2024-07-18",
  "object": "response",
  "output": [
    {
      "id": "msg_0e8c5c6bc6024a90006ab4b9872d2487d2bbc20b9e641ef848",
      "content": [
        {
          "annotations": [],
          "text": "Once upon a time, in a land where rainbows twisted through the sky, lived a unicorn named Sparkle. Unlike most unicorns, Sparkle was known for her clumsiness. Her dazzling horn often got her into trouble, causing mischief instead of magic.\n\nOne sunny day, Sparkle decided to explore the Enchanted Forest, a place filled with wonders and mysteries. Eager to impress her friends, she aimed to create a beautiful flower crown for herself. As she trotted through the meadows, she lost her footing on a patch of slippery moss. She stumbled and fell, accidentally knocking over a beehive.

In [42]:
usage = response.usage

cost = calculate_cost(
    model_name="gpt-4o-mini",
    input_tokens=usage.input_tokens,
    output_tokens=usage.output_tokens
)

print(f"Cost: ${cost:.6f}")

Cost: $0.000236


System prompts to give instructions to the model

In [43]:
instructions = "make sure to add emojis"
user_prompt = "tell me a bad time story about a unicorn"

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": user_prompt},
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages
)

In [44]:
print(response.output_text)

Once upon a time in a glittery forest 🌳, there lived a unicorn named Sparkle. She had a beautiful, shimmering horn that glowed like the stars 🌟. Everyone in the forest adored her because she brought joy wherever she went. But one day, Sparkle woke up feeling grumpy. 😠

Instead of spreading happiness, she decided to play a prank on her friends. First, she sprinkled glitter over the sleeping fairies 🧚‍♀️, making them sparkle uncontrollably and dance around in confusion. The fairies giggled at first, but soon they were so dizzy that they couldn't fly straight!

Next, Sparkle sneaked up on the wise old owl 🦉 and hooted loudly, startling him so much that he flew right into a bush. Everyone in the forest laughed, but the owl didn't seem amused. Instead, he just shook his head and said, "Not every joke is funny, Sparkle."

Feeling a little guilty, Sparkle trotted off to the river to clear her mind. As she gazed at her reflection, she noticed that her beautiful horn wasn’t shining. 🦄✨ Suddenly

Los LLM son stateless. Primer veremos como no retienen el estado entre llamadas

In [45]:
instructions = "make sure to add emojis"
user_prompt = "my name is Alexey"

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": user_prompt},
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages
)

print(response.output_text)

Nice to meet you, Alexey! 😊 What would you like to talk about today?


ahora volvemos a preguntar por nuestro nombre

In [46]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input="What's my name?"
)

print(response.output_text)

I don’t have access to personal data about you unless you share it with me. If you let me know your name, I can remember it for our conversation!


Para mantener el estado se debe mandar la conversación entera en cada turno.

In [47]:
instructions = "make sure to add emojis"
user_prompt = "my name is Alexey"

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": user_prompt},
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages
)

print(response.output_text)

Nice to meet you, Alexey! 😊 How can I assist you today?


Añadimos la respuesta a la lista de mensajes

In [48]:
messages.extend(response.output)

In [49]:
print(messages)

[{'role': 'system', 'content': 'make sure to add emojis'}, {'role': 'user', 'content': 'my name is Alexey'}, ResponseOutputMessage(id='msg_04c4f0755e1ea92a006ab4ba9593b487d282f33a3bc883f17a', content=[ResponseOutputText(annotations=[], text='Nice to meet you, Alexey! 😊 How can I assist you today?', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]


Añadimos la pregunta también a la lista de mensajes

In [50]:
messages.append(
    {"role": "user", "content": "what's my name?"}
)

In [51]:
print(messages)

[{'role': 'system', 'content': 'make sure to add emojis'}, {'role': 'user', 'content': 'my name is Alexey'}, ResponseOutputMessage(id='msg_04c4f0755e1ea92a006ab4ba9593b487d282f33a3bc883f17a', content=[ResponseOutputText(annotations=[], text='Nice to meet you, Alexey! 😊 How can I assist you today?', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None), {'role': 'user', 'content': "what's my name?"}]


Y mandamos toda la conversación

In [52]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages
)

print(response.output_text)

Your name is Alexey! 😊


Por esto mismo hay que tener cuidado porque podemos mandar ok o vale y hacer que el LLM tenga que recibir toda la conversación para contestarnos solo "muy bien"